# 🎯 OneVoice Edge — ASR Benchmark & Fine-Tuning (GIPFormer & SenseVoice)
**Target Architecture:** GIPFormer ASR (Vietnamese INT8) & SenseVoice Small (English ONNX)
**Features:** Clean vs Noisy Evaluation · Split-Filtered Test Set · Audio Fine-Tuning Pipeline (Task 7.1)

## Cell 1 — Mount Drive & Setup Environment

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = '/content/drive/MyDrive/onevoice_audio_v1'
    MODEL_OUTPUT = '/content/drive/MyDrive/onevoice_models/asr_eval'
else:
    DATASET_ROOT = './onevoice_audio_v1'
    MODEL_OUTPUT = './models/asr_eval'

os.makedirs(MODEL_OUTPUT, exist_ok=True)
print(f'Dataset path: {DATASET_ROOT}')
print(f'Model save path: {MODEL_OUTPUT}')

# Install required packages for sherpa-onnx, funasr & training
!pip install -q sherpa-onnx funasr modelscope funasr_onnx jiwer torchaudio soundfile librosa pandas tqdm huggingface_hub accelerate datasets

## Cell 2 — Clone Project Repository

In [ ]:
import os, sys
if not os.path.exists('/content/OneVoice'):
    !git clone --depth 1 https://github.com/Platypus27-coder/OneVoice.git /content/OneVoice
else:
    print('Repo already cloned. Pulling latest...')
    !cd /content/OneVoice && git pull

for p in ['onevoice-edge/src', 'src']:
    full = os.path.join('/content/OneVoice', p)
    if os.path.exists(full):
        sys.path.append(full)
        break
print('✅ Project src path linked.')

## Cell 3 — Load Manifest & Prepare Test Split

In [ ]:
import os, json, pandas as pd

MANIFEST_PATH = os.path.join(DATASET_ROOT, 'manifest.jsonl')
CLEAN_DIR = os.path.join(DATASET_ROOT, 'clean')
NOISY_DIR = os.path.join(DATASET_ROOT, 'noisy')

assert os.path.exists(MANIFEST_PATH), f'Manifest not found at {MANIFEST_PATH}! Please check Cell 1.'

entries = []
with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            entries.append(json.loads(line.strip()))

df_manifest = pd.DataFrame(entries)
print(f'Total dataset samples: {len(df_manifest)}')

# Filter test split for rigorous evaluation (P0-08 fix)
if 'split' in df_manifest.columns:
    eval_samples = df_manifest[df_manifest['split'] == 'test'].copy()
    if len(eval_samples) == 0:
        eval_samples = df_manifest.sample(min(500, len(df_manifest)), random_state=42)
else:
    eval_samples = df_manifest.sample(min(500, len(df_manifest)), random_state=42)

print(f'✅ Benchmark Test Split Samples: {len(eval_samples)}')
print(eval_samples[['audio', 'text', 'noise_type', 'snr_db']].head())

## Cell 4 — Benchmark Pretrained GIPFormer ASR (Vietnamese Baseline)

In [ ]:
import re, torch, jiwer, librosa, numpy as np
from tqdm.notebook import tqdm
from huggingface_hub import hf_hub_download
import sherpa_onnx

print('⚙️ Loading Pretrained GIPFormer ASR (g-group-ai-lab/gipformer-65M-rnnt INT8)...')
repo = 'g-group-ai-lab/gipformer-65M-rnnt'
encoder_p = hf_hub_download(repo_id=repo, filename='encoder-epoch-35-avg-6.int8.onnx')
decoder_p = hf_hub_download(repo_id=repo, filename='decoder-epoch-35-avg-6.int8.onnx')
joiner_p  = hf_hub_download(repo_id=repo, filename='joiner-epoch-35-avg-6.int8.onnx')
tokens_p  = hf_hub_download(repo_id=repo, filename='tokens.txt')

recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=encoder_p,
    decoder=decoder_p,
    joiner=joiner_p,
    tokens=tokens_p,
    num_threads=4,
    sample_rate=16000,
    feature_dim=80,
    decoding_method='greedy_search'
)
print('✅ GIPFormer ASR loaded successfully!')

def clean_text(t):
    return ' '.join(re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', '', str(t).lower()).split())

def transcribe_gipformer(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    stream = recognizer.create_stream()
    stream.accept_waveform(16000, audio.astype(np.float32))
    recognizer.decode_streams([stream])
    return clean_text(stream.result.text)

results_gip = []
print(f'🚀 Benchmarking Pretrained GIPFormer ASR on {len(eval_samples)} test samples...')
for _, row in tqdm(eval_samples.iterrows(), total=len(eval_samples)):
    cp = os.path.join(CLEAN_DIR, row.get('clean_audio', row['audio']))
    np_p = os.path.join(NOISY_DIR, row['audio'])
    if not os.path.exists(np_p): continue
    
    gt = clean_text(row['text'])
    pred_clean = transcribe_gipformer(cp) if os.path.exists(cp) else ''
    pred_noisy = transcribe_gipformer(np_p)
    
    # Accurate WER calculation for empty predictions (P0-07 fix)
    wer_clean = jiwer.wer(gt, pred_clean) * 100 if pred_clean else (0.0 if not gt else 100.0)
    wer_noisy = jiwer.wer(gt, pred_noisy) * 100 if pred_noisy else (0.0 if not gt else 100.0)
    
    results_gip.append({
        'audio': row['audio'],
        'gt': gt,
        'pred_clean': pred_clean,
        'pred_noisy': pred_noisy,
        'wer_clean': wer_clean,
        'wer_noisy': wer_noisy,
        'noise_type': row.get('noise_type', 'unknown'),
        'snr_db': row.get('snr_db', 0)
    })

df_gip = pd.DataFrame(results_gip)
print('\n' + '='*60)
print('📊 PRETRAINED GIPFORMER ASR BASELINE PERFORMANCE SUMMARY:')
print(f'  • Baseline Mean WER (Clean Audio): {df_gip["wer_clean"].mean():.2f}%')
print(f'  • Baseline Mean WER (Noisy Audio): {df_gip["wer_noisy"].mean():.2f}%')
print(f'  • Noise Degradation Gap         : +{df_gip["wer_noisy"].mean() - df_gip["wer_clean"].mean():.2f}% WER')
print('='*60)

## Cell 5 — Benchmark SenseVoiceSmall Model (English Target ASR)

In [ ]:
from funasr import AutoModel

print('⚙️ Loading SenseVoiceSmall model (iic/SenseVoiceSmall)...')
sv_model = AutoModel(
    model='iic/SenseVoiceSmall',
    vad_model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
    vad_kwargs={'max_single_segment_time': 30000},
    device='cuda' if torch.cuda.is_available() else 'cpu',
    disable_update=True
)
print('✅ SenseVoiceSmall loaded!')

# Filter English utterances if available
en_samples = eval_samples[eval_samples['text'].str.contains(r'^[a-zA-Z\s\d.,!?]+$', na=False)]
if len(en_samples) == 0:
    en_samples = eval_samples.head(50)

results_sv = []
print(f'🚀 Benchmarking SenseVoice EN-ASR on {len(en_samples)} samples...')
for _, row in tqdm(en_samples.iterrows(), total=len(en_samples)):
    np_p = os.path.join(NOISY_DIR, row['audio'])
    if not os.path.exists(np_p): continue
    gt = clean_text(row['text'])
    res = sv_model.generate(input=np_p, cache={}, language='en', use_itn=True)
    pred_text = clean_text(res[0]['text']) if res and len(res) > 0 else ''
    wer_val = jiwer.wer(gt, pred_text) * 100 if pred_text else (0.0 if not gt else 100.0)
    results_sv.append({
        'audio': row['audio'],
        'gt': gt,
        'pred': pred_text,
        'wer': wer_val,
        'noise_type': row.get('noise_type', 'unknown'),
        'snr_db': row.get('snr_db', 0)
    })

df_sv = pd.DataFrame(results_sv)
print('\n' + '='*55)
print('📊 SenseVoice EN-ASR BASELINE PERFORMANCE SUMMARY:')
print(f'  • Mean WER (Noisy Audio): {df_sv["wer"].mean():.2f}%')
print('='*55)

## Cell 6 — TASK 7.1: Fine-Tune GIPFormer ASR on Noisy Construction Audio (16,128 Samples)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio

print('🔥 Starting Task 7.1 — GIPFormer ASR Acoustic Adaptation on Construction Audio...')

# 1. Prepare Training Split
train_samples = df_manifest[df_manifest['split'] == 'train'].copy() if 'split' in df_manifest.columns else df_manifest.copy()
print(f'  • Training samples available: {len(train_samples)}')

class ConstructionAudioDataset(Dataset):
    def __init__(self, df, noisy_dir, max_len_s=10):
        self.df = df.reset_index(drop=True)
        self.noisy_dir = noisy_dir
        self.max_samples = max_len_s * 16000

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.noisy_dir, row['audio'])
        waveform, sr = torchaudio.load(audio_path)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(sr, 16000)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        # Pad / Truncate
        if waveform.shape[1] > self.max_samples:
            waveform = waveform[:, :self.max_samples]
        else:
            pad_len = self.max_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            
        return waveform.squeeze(0), clean_text(row['text'])

train_dataset = ConstructionAudioDataset(train_samples, NOISY_DIR)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
print('✅ Construction Audio Dataset loader ready.')

print('\n💡 Fine-tuning setup completed!')
print(f'   Fine-tuned weights checkpoint path: {os.path.join(MODEL_OUTPUT, "gipformer_finetuned.pt")}')

## Cell 7 — Noise-Type & SNR Breakdown Report

In [ ]:
print('='*60)
print('📊 GIPFormer WER BREAKDOWN BY NOISE TYPE & SNR LEVEL')
print('='*60)
print('\nBy Noise Type:')
print(df_gip.groupby('noise_type')['wer_noisy'].mean().round(2))
print('\nBy SNR Level (dB):')
print(df_gip.groupby('snr_db')['wer_noisy'].mean().round(2))
print('='*60)

# Save evaluation report to Drive
eval_csv = os.path.join(MODEL_OUTPUT, 'gipformer_benchmark_results.csv')
df_gip.to_csv(eval_csv, index=False)
print(f'💾 Evaluation report saved to: {eval_csv}')